# **SPOTIFY DATA ANALYTICS CAPSTONE**
## **By Prachi Sharma**
#### A comprehensive analysis of Spotify's music data to uncover insights about music trends, user preferences, and song characteristics — covering 170,000+ tracks spanning from 1921 to 2020.

## 1: INTODUCTION
#### 1.1 Project Overview & Objectives
#### The music industry generates vast amounts of data through streaming platforms like Spotify. This capstone project performs a comprehensive analysis of Spotify's music catalog to:

#### Understand music characteristics — Analyze audio features like danceability, energy, valence, and tempo across tracks
#### Identify historical trends — Track how music has evolved over the past century (1921–2020)
#### Discover genre patterns — Compare audio features across nearly 3,000 music genres
#### Analyze artist profiles — Examine how artists differ in their musical fingerprints
#### Build predictive models — Forecast song popularity using audio features and machine learning

## 2. DATA COLLECTION & PREPROCESSING

## 2.1 Load the Datasets

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import altair as alt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

alt.data_transformers.disable_max_rows()

In [ ]:
# Load All 5 Datasets 
df_tracks = pd.read_csv(r"C:\Users\Prachi sharma\Downloads\data.csv")
df_genres = pd.read_csv(r"C:\Users\Prachi sharma\Downloads\data_w_genres.csv")
df_by_year = pd.read_csv(r"C:\Users\Prachi sharma\Downloads\data_by_year.csv")
df_by_artist = pd.read_csv(r"C:\Users\Prachi sharma\Downloads\data_by_artist.csv")
df_by_genres = pd.read_csv(r"C:\Users\Prachi sharma\Downloads\data_by_genres.csv")

print(f"Tracks dataset: {df_tracks.shape[0]:,} rows x {df_tracks.shape[1]} columns")
print(f"Genres dataset: {df_genres.shape[0]:,} rows x {df_genres.shape[1]} columns")
print(f"By Year dataset: {df_by_year.shape[0]:,} rows x {df_by_year.shape[1]} columns")
print(f"By Artist dataset: {df_by_artist.shape[0]:,} rows x {df_by_artist.shape[1]} columns")
print(f"By Genres dataset: {df_by_genres.shape[0]:,} rows x {df_by_genres.shape[1]} columns")

In [ ]:
df_tracks.head()

In [ ]:
df_tracks.info()

## 2.2 Inspect and Clean the Data

In [ ]:
# Check missing values across all datasets
datasets = {
    'Tracks': df_tracks,
    'Genres': df_genres,
    'By Year': df_by_year,
    'By Artist': df_by_artist,
    'By Genres': df_by_genres
}

missing_summary = []
for name, df in datasets.items():
    total_missing = df.isnull().sum().sum()
    total_cells = df.shape[0] * df.shape[1]
    missing_summary.append({
        'Dataset': name,
        'Total Cells': f"{total_cells:,}",
        'Missing Values': total_missing,
        'Missing %': f"{(total_missing/total_cells)*100:.2f}%"
    })

pd.DataFrame(missing_summary)

In [ ]:
# Check for duplicate tracks by ID
duplicates_by_id = df_tracks.duplicated(subset=['id']).sum()
duplicates_full = df_tracks.duplicated().sum()
print(f"Duplicate track IDs: {duplicates_by_id:,}")
print(f"Fully duplicate rows: {duplicates_full:,}")

# Remove duplicate tracks
df_tracks = df_tracks.drop_duplicates(subset=['id'], keep='first')
print(f"\nAfter removing duplicates: {df_tracks.shape[0]:,} tracks remain")

In [ ]:
# Check and fix data types
# Convert 'explicit' to boolean
df_tracks['explicit'] = df_tracks['explicit'].astype(bool)

# Extract year from release_date
df_tracks['release_year'] = df_tracks['release_date'].apply(
    lambda x: int(str(x)[:4]) if pd.notna(x) else np.nan
)

# Convert duration from ms to minutes
df_tracks['duration_min'] = df_tracks['duration_ms'] / 60000

# Clean artists column— convert string representation of list to actual list
df_tracks['artists_clean'] = df_tracks['artists'].apply(
    lambda x: ', '.join(eval(x)) if pd.notna(x) else 'Unknown'
)

print("Data types after cleaning:")
df_tracks[['explicit', 'release_year', 'duration_min']].dtypes

In [ ]:
# Check for outliers in key numeric features
numeric_features = ['danceability', 'energy', 'valence', 'acousticness', 
                    'instrumentalness', 'liveness', 'speechiness', 'tempo', 
                    'loudness', 'popularity', 'duration_min']

outlier_report = []
for feat in numeric_features:
    Q1 = df_tracks[feat].quantile(0.25)
    Q3 = df_tracks[feat].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_outliers = ((df_tracks[feat] < lower) | (df_tracks[feat] > upper)).sum()
    outlier_report.append({
        'Feature': feat,
        'Min': f"{df_tracks[feat].min():.3f}",
        'Max': f"{df_tracks[feat].max():.3f}",
        'Mean': f"{df_tracks[feat].mean():.3f}",
        'Outliers': f"{n_outliers:,}",
        'Outlier %': f"{(n_outliers/len(df_tracks))*100:.1f}%"
    })

pd.DataFrame(outlier_report)

In [ ]:
# Remove extreme duration outliers (< 15 seconds or > 60 minutes — likely errors)
before = len(df_tracks)
df_tracks = df_tracks[(df_tracks['duration_min'] >= 0.25) & (df_tracks['duration_min'] <= 60)]
print(f"Removed {before - len(df_tracks):,} tracks with extreme durations")
print(f"Clean dataset: {df_tracks.shape[0]:,} tracks")

## 2.3 Exploratory Data Analysis (EDA)

In [ ]:
# Descriptive statistics for key audio features
audio_features = ['danceability', 'energy', 'valence', 'acousticness', 
                  'instrumentalness', 'liveness', 'speechiness', 'tempo', 
                  'loudness', 'popularity']

df_tracks[audio_features].describe().round(3)

In [ ]:
# Year distribution of tracks
year_counts = df_tracks['release_year'].value_counts().sort_index().reset_index()
year_counts.columns = ['Year', 'Count']

chart = alt.Chart(year_counts).mark_area(
    interpolate='monotone', fillOpacity=0.6, line=True
).encode(
    x=alt.X('Year:Q', title='Release Year', scale=alt.Scale(domain=[1920, 2021])),
    y=alt.Y('Count:Q', title='Number of Tracks'),
    tooltip=['Year:Q', 'Count:Q']
).properties(
    title=alt.Title('Distribution of Tracks by Release Year',
                    subtitle='Exponential growth in music production since the 1960s'),
    height=350
)
chart

In [ ]:
# Popularity distribution
pop_hist = alt.Chart(df_tracks.sample(20000, random_state=42)).mark_bar(
    cornerRadiusTopLeft=2, cornerRadiusTopRight=2
).encode(
    x=alt.X('popularity:Q', bin=alt.Bin(maxbins=50), title='Popularity Score'),
    y=alt.Y('count():Q', title='Number of Tracks'),
    tooltip=['count():Q']
).properties(
    title=alt.Title('Distribution of Track Popularity',
                    subtitle='Most tracks have low popularity — a classic long-tail distribution'),
    height=350
)
pop_hist

In [ ]:
# Explicit vs non-explicit content ratio
explicit_counts = df_tracks['explicit'].value_counts().reset_index()
explicit_counts.columns = ['Explicit', 'Count']
explicit_counts['Label'] = explicit_counts['Explicit'].map({True: 'Explicit', False: 'Non-Explicit'})
explicit_counts['Percentage'] = (explicit_counts['Count'] / explicit_counts['Count'].sum() * 100).round(1)

chart = alt.Chart(explicit_counts).mark_arc(innerRadius=60).encode(
    theta=alt.Theta('Count:Q'),
    color=alt.Color('Label:N', scale=alt.Scale(range=['#1DB954', '#b3b3b3']),
                    legend=alt.Legend(title='Content Type')),
    tooltip=['Label:N', 'Count:Q', 'Percentage:Q']
).properties(
    title=alt.Title('Explicit vs Non-Explicit Tracks'),
    height=300
)
chart

In [ ]:
# Top 15 most popular tracks
top_tracks = df_tracks.nlargest(15, 'popularity')[
    ['name', 'artists_clean', 'popularity', 'release_year']
].copy()
top_tracks.columns = ['Track Name', 'Artist(s)', 'Popularity', 'Year']
top_tracks = top_tracks.reset_index(drop=True)
top_tracks.index = top_tracks.index + 1
top_tracks

## 3. DATA ANALYSIS

## 3.1 Distribution of Audio Features
#### We analyze how key audio features are distributed across all tracks to understand the overall musical landscape on Spotify.

In [ ]:
norm_features = ['danceability', 'energy', 'valence', 'acousticness', 
                 'instrumentalness', 'liveness', 'speechiness']

sample_df = df_tracks[norm_features].sample(20000, random_state=42).melt(
    var_name='Feature', value_name='Value'
)

chart = alt.Chart(sample_df).mark_area(
    interpolate='monotone', fillOpacity=0.5, line=True
).encode(
    x=alt.X('Value:Q', bin=alt.Bin(maxbins=40), title='Feature Value (0–1)'),
    y=alt.Y('count():Q', title='Frequency', stack=None),
    color=alt.Color('Feature:N', legend=alt.Legend(title='Audio Feature')),
    tooltip=['Feature:N', 'count():Q']
).properties(
    title=alt.Title('Distribution of Audio Features',
                    subtitle='Each feature shows a distinct pattern — energy and danceability cluster around 0.5–0.7'),
    height=400
)
chart

In [ ]:
# Tempo distribution
# Tempo
tempo_chart = alt.Chart(df_tracks.sample(20000, random_state=42)).mark_bar(
    cornerRadiusTopLeft=2, cornerRadiusTopRight=2
).encode(
    x=alt.X('tempo:Q', bin=alt.Bin(maxbins=50), title='Tempo (BPM)'),
    y=alt.Y('count():Q', title='Number of Tracks'),
    tooltip=['count():Q']
).properties(
    title=alt.Title('Distribution of Tempo',
                    subtitle='Most tracks fall between 80–140 BPM, with a peak around 120 BPM'),
    height=350
)
tempo_chart

In [ ]:
# Loudness distribution
loud_chart = alt.Chart(df_tracks.sample(20000, random_state=42)).mark_bar(
    cornerRadiusTopLeft=2, cornerRadiusTopRight=2
).encode(
    x=alt.X('loudness:Q', bin=alt.Bin(maxbins=50), title='Loudness (dB)'),
    y=alt.Y('count():Q', title='Number of Tracks'),
    tooltip=['count():Q']
).properties(
    title=alt.Title('Distribution of Loudness',
                    subtitle='Centered around -7 dB, reflecting the loudness war in modern production'),
    height=350
)
loud_chart

#### Key Observations:

##### Danceability is roughly normally distributed, centered around 0.55 — most music is moderately danceable
##### Energy has a slight right skew, with a peak around 0.6–0.8
##### Instrumentalness and speechiness are heavily right-skewed — most tracks have vocals and minimal spoken word
##### Acousticness is bimodal — tracks tend to be either very acoustic or not acoustic at all
##### Tempo clusters between 80–140 BPM, standard for popular music
##### Loudness centers around -7 dB, reflecting modern production standards

## 3.2 Trends Over Time
#### Using the year-level aggregated data, we examine how music characteristics have evolved from 1921 to 2020.

In [ ]:
# Trends in key audio features over time
trend_features = ['acousticness', 'danceability', 'energy', 'valence', 'instrumentalness']

trend_data = df_by_year[['year'] + trend_features].melt(
    id_vars='year', var_name='Feature', value_name='Value'
)

chart = alt.Chart(trend_data).mark_line(strokeWidth=2.5).encode(
    x=alt.X('year:Q', title='Year', scale=alt.Scale(domain=[1920, 2021])),
    y=alt.Y('Value:Q', title='Feature Value'),
    color=alt.Color('Feature:N', legend=alt.Legend(title='Audio Feature')),
    tooltip=['year:Q', 'Feature:N', alt.Tooltip('Value:Q', format='.3f')]
).properties(
    title=alt.Title('Evolution of Audio Features (1921–2020)',
                    subtitle='Acousticness declined dramatically while energy and danceability rose'),
    height=400
)
chart

In [ ]:
# Loudness and temp trend
loudness_line = alt.Chart(df_by_year).mark_line(strokeWidth=2.5, color='#e74c3c').encode(
    x=alt.X('year:Q', title='Year', scale=alt.Scale(domain=[1920, 2021])),
    y=alt.Y('loudness:Q', title='Average Loudness (dB)'),
    tooltip=['year:Q', alt.Tooltip('loudness:Q', format='.1f')]
).properties(
    title=alt.Title('Average Loudness Over Time',
                    subtitle='The "Loudness War": music has gotten progressively louder since the 1960s'),
    height=350
)
loudness_line

In [ ]:
# Tempo trend over time
tempo_line = alt.Chart(df_by_year).mark_line(strokeWidth=2.5, color='#3498db').encode(
    x=alt.X('year:Q', title='Year', scale=alt.Scale(domain=[1920, 2021])),
    y=alt.Y('tempo:Q', title='Average Tempo (BPM)'),
    tooltip=['year:Q', alt.Tooltip('tempo:Q', format='.1f')]
).properties(
    title=alt.Title('Average Tempo Over Time',
                    subtitle='Tempo has fluctuated but shows a general upward trend in recent decades'),
    height=350
)
tempo_line

In [ ]:
# Popularity trend over time
pop_line = alt.Chart(df_by_year).mark_line(strokeWidth=2.5, color='#1DB954').encode(
    x=alt.X('year:Q', title='Year', scale=alt.Scale(domain=[1920, 2021])),
    y=alt.Y('popularity:Q', title='Average Popularity'),
    tooltip=['year:Q', alt.Tooltip('popularity:Q', format='.1f')]
).properties(
    title=alt.Title('Average Track Popularity by Release Year',
                    subtitle="Recent music dominates popularity — Spotify's algorithm favors new releases"),
    height=350
)
pop_line

#### Key Trends:

##### Acousticness has plummeted from ~0.9 (1920s) to ~0.2 (2020) — reflecting the shift to electronic and amplified music
##### Energy has risen dramatically — modern tracks are much more energetic than early recordings
##### Danceability has increased gradually, especially since the 1980s with the rise of pop and electronic music
##### Valence (happiness) peaked in the 1960s and has been declining — modern music tends to be more melancholic
##### The Loudness War is clearly visible — average loudness increased from -25 dB to about -7 dB
##### Popularity is heavily biased toward recent music on Spotify

## 3.3 Correlations Between Features
#### We examine how audio features relate to each other and to popularity.

In [ ]:
# Correlation matrix heatmap
corr_features = ['danceability', 'energy', 'valence', 'acousticness', 
                 'instrumentalness', 'liveness', 'speechiness', 'tempo', 
                 'loudness', 'popularity']

corr_matrix = df_tracks[corr_features].corr().round(3)

# Melt for Altair heatmap
corr_melted = corr_matrix.reset_index().melt(
    id_vars='index', var_name='Feature_2', value_name='Correlation'
)
corr_melted.columns = ['Feature_1', 'Feature_2', 'Correlation']

heatmap = alt.Chart(corr_melted).mark_rect().encode(
    x=alt.X('Feature_1:N', title=None, sort=corr_features),
    y=alt.Y('Feature_2:N', title=None, sort=corr_features),
    color=alt.Color('Correlation:Q', scale=alt.Scale(scheme='redblue', domain=[-1, 1]),
                    legend=alt.Legend(title='Correlation')),
    tooltip=['Feature_1:N', 'Feature_2:N', alt.Tooltip('Correlation:Q', format='.3f')]
).properties(
    title=alt.Title('Correlation Matrix of Audio Features',
                    subtitle='Energy-loudness are strongly correlated; acousticness opposes energy'),
    height=400, width=500
)

text = alt.Chart(corr_melted).mark_text(fontSize=10).encode(
    x=alt.X('Feature_1:N', sort=corr_features),
    y=alt.Y('Feature_2:N', sort=corr_features),
    text=alt.Text('Correlation:Q', format='.2f'),
    color=alt.condition(
        (alt.datum.Correlation > 0.5) | (alt.datum.Correlation < -0.5),
        alt.value('white'),
        alt.value('black')
    )
)

heatmap + text

In [ ]:
# Energy vs Danceability scatter
sample = df_tracks.sample(5000, random_state=42)

# Energy vs Danceability
scatter = alt.Chart(sample).mark_circle(size=15, opacity=0.4).encode(
    x=alt.X('energy:Q', title='Energy'),
    y=alt.Y('danceability:Q', title='Danceability'),
    color=alt.Color('popularity:Q', scale=alt.Scale(scheme='viridis'),
                    legend=alt.Legend(title='Popularity')),
    tooltip=['name:N', 'artists_clean:N', 'energy:Q', 'danceability:Q', 'popularity:Q']
).properties(
    title=alt.Title('Energy vs Danceability',
                    subtitle='Most popular tracks have moderate-to-high energy and high danceability'),
    height=400
)
scatter

In [ ]:
# Energy vs Acousticness -strong negative correlation
scatter2 = alt.Chart(sample).mark_circle(size=15, opacity=0.4).encode(
    x=alt.X('energy:Q', title='Energy'),
    y=alt.Y('acousticness:Q', title='Acousticness'),
    color=alt.Color('popularity:Q', scale=alt.Scale(scheme='viridis'),
                    legend=alt.Legend(title='Popularity')),
    tooltip=['name:N', 'artists_clean:N', 'energy:Q', 'acousticness:Q', 'popularity:Q']
).properties(
    title=alt.Title('Energy vs Acousticness',
                    subtitle='Strong inverse relationship — high-energy tracks are rarely acoustic'),
    height=400
)
scatter2

In [ ]:
# Valence vs Energy (Emotional Mapping)
scatter3 = alt.Chart(sample).mark_circle(size=15, opacity=0.4).encode(
    x=alt.X('valence:Q', title='Valence (Happiness)'),
    y=alt.Y('energy:Q', title='Energy'),
    color=alt.Color('popularity:Q', scale=alt.Scale(scheme='viridis'),
                    legend=alt.Legend(title='Popularity')),
    tooltip=['name:N', 'artists_clean:N', 'valence:Q', 'energy:Q', 'popularity:Q']
).properties(
    title=alt.Title('Emotional Mapping: Valence vs Energy',
                    subtitle='Music spans all emotional quadrants — happy/energetic to sad/calm'),
    height=400
)
scatter3

#### *Correlation Insights*:

##### Energy ↔ Loudness (r ≈ 0.76): Strongest positive correlation — louder tracks are perceived as more energetic
##### Energy ↔ Acousticness (r ≈ -0.73): Strongest negative correlation — electronic/amplified = energetic, acoustic = calm
##### Danceability ↔ Valence (r ≈ 0.43): Happy music is more danceable
##### Popularity has weak correlations with audio features — popularity depends on many external factors (artist fame, marketing, playlist placement)

## 3.4 User Preferences & Listening Habits
#### We analyze what makes tracks popular and how genres differ in their audio profiles.

In [ ]:
# Popular vs Unpopular tracks comparison
df_tracks['popularity_tier'] = pd.cut(
    df_tracks['popularity'], 
    bins=[0, 20, 50, 75, 100], 
    labels=['Low (0-20)', 'Medium (21-50)', 'High (51-75)', 'Very High (76-100)']
)

tier_means = df_tracks.groupby('popularity_tier', observed=True)[norm_features].mean().reset_index()
tier_melted = tier_means.melt(id_vars='popularity_tier', var_name='Feature', value_name='Mean Value')

chart = alt.Chart(tier_melted).mark_bar().encode(
    x=alt.X('Feature:N', title=None),
    y=alt.Y('Mean Value:Q', title='Average Value'),
    color=alt.Color('popularity_tier:N', legend=alt.Legend(title='Popularity Tier')),
    xOffset='popularity_tier:N',
    tooltip=['popularity_tier:N', 'Feature:N', alt.Tooltip('Mean Value:Q', format='.3f')]
).properties(
    title=alt.Title('Audio Feature Profiles by Popularity Tier',
                    subtitle='Popular tracks tend to be more danceable, energetic, and less acoustic'),
    height=400
)
chart

In [ ]:
# Top 15 genres by popularity
df_by_genres_clean = df_by_genres[df_by_genres['genres'] != '[]'].copy()
top_genres = df_by_genres_clean.nlargest(15, 'popularity')[
    ['genres', 'popularity', 'danceability', 'energy', 'valence']
].copy()

chart = alt.Chart(top_genres).mark_bar(
    cornerRadiusTopLeft=4, cornerRadiusTopRight=4
).encode(
    x=alt.X('popularity:Q', title='Average Popularity'),
    y=alt.Y('genres:N', title=None, sort='-x'),
    color=alt.Color('danceability:Q', scale=alt.Scale(scheme='viridis'),
                    legend=alt.Legend(title='Danceability')),
    tooltip=['genres:N', 'popularity:Q', 'danceability:Q', 'energy:Q', 'valence:Q']
).properties(
    title=alt.Title('Top 15 Genres by Popularity',
                    subtitle='Pop and reggaeton dominate — high danceability is a common trait'),
    height=400
)
chart

In [ ]:
# Genre audio feature radar-style comparison (as grouped bar chart)
selected_genres = ['pop', 'rock', 'hip hop', 'jazz', 'classical', 
                   'electronic', 'r&b', 'country', 'metal', 'latin']
genre_profiles = df_by_genres_clean[df_by_genres_clean['genres'].isin(selected_genres)].copy()

genre_melted = genre_profiles[['genres', 'danceability', 'energy', 'valence', 
                                'acousticness', 'instrumentalness', 'speechiness']].melt(
    id_vars='genres', var_name='Feature', value_name='Value'
)

chart = alt.Chart(genre_melted).mark_bar().encode(
    x=alt.X('Value:Q', title='Feature Value'),
    y=alt.Y('genres:N', title=None, sort=selected_genres),
    color=alt.Color('Feature:N', legend=alt.Legend(title='Feature')),
    row=alt.Row('Feature:N', title=None, 
                header=alt.Header(labelAngle=0, labelAlign='left'))
).properties(
    title=alt.Title('Audio Feature Profiles Across Popular Genres'),
    height=80, width=400
)
chart

In [ ]:
# Explicit content trend over decades
df_tracks['decade'] = (df_tracks['release_year'] // 10) * 10
explicit_by_decade = df_tracks.groupby('decade')['explicit'].mean().reset_index()
explicit_by_decade.columns = ['Decade', 'Explicit Ratio']
explicit_by_decade = explicit_by_decade[explicit_by_decade['Decade'] >= 1950]

chart = alt.Chart(explicit_by_decade).mark_bar(
    cornerRadiusTopLeft=4, cornerRadiusTopRight=4
).encode(
    x=alt.X('Decade:O', title='Decade'),
    y=alt.Y('Explicit Ratio:Q', title='Proportion of Explicit Tracks',
            axis=alt.Axis(format='%')),
    tooltip=['Decade:O', alt.Tooltip('Explicit Ratio:Q', format='.1%')]
).properties(
    title=alt.Title('Rise of Explicit Content Over Decades',
                    subtitle='Dramatic increase starting in the 1990s with hip-hop and modern pop'),
    height=350
)
chart

#### *User Preference Insights*:

##### Popular tracks are more danceable, louder, and less acoustic — listeners prefer energetic, polished production
##### Pop and reggaeton lead in popularity, both featuring high danceability
##### Genre fingerprints are distinct — classical has high acousticness/instrumentalness, hip hop has high speechiness, metal has high energy
##### Explicit content has surged since the 1990s, now comprising a significant portion of popular music

## 4. VISUALIZATION

## 4.1 Key Findings Visualizations

In [ ]:
# Top 20 most prolific artists by track count
artist_counts = df_by_artist.nlargest(20, 'count')[['artists', 'count', 'popularity']].copy()

chart = alt.Chart(artist_counts).mark_bar(
    cornerRadiusTopLeft=4, cornerRadiusTopRight=4
).encode(
    x=alt.X('count:Q', title='Number of Tracks'),
    y=alt.Y('artists:N', title=None, sort='-x'),
    color=alt.Color('popularity:Q', scale=alt.Scale(scheme='viridis'),
                    legend=alt.Legend(title='Avg Popularity')),
    tooltip=['artists:N', 'count:Q', alt.Tooltip('popularity:Q', format='.1f')]
).properties(
    title=alt.Title('Top 20 Most Prolific Artists on Spotify',
                    subtitle='Track count does not guarantee high popularity'),
    height=500
)
chart

In [ ]:
# Musical key distribution
key_names = {0: 'C', 1: 'C#', 2: 'D', 3: 'D#', 4: 'E', 5: 'F', 
             6: 'F#', 7: 'G', 8: 'G#', 9: 'A', 10: 'A#', 11: 'B'}

key_dist = df_tracks['key'].value_counts().sort_index().reset_index()
key_dist.columns = ['Key', 'Count']
key_dist['Key Name'] = key_dist['Key'].map(key_names)

chart = alt.Chart(key_dist).mark_bar(
    cornerRadiusTopLeft=4, cornerRadiusTopRight=4
).encode(
    x=alt.X('Key Name:N', title='Musical Key', sort=list(key_names.values())),
    y=alt.Y('Count:Q', title='Number of Tracks'),
    tooltip=['Key Name:N', 'Count:Q']
).properties(
    title=alt.Title('Distribution of Musical Keys',
                    subtitle='G, C, and D are the most common keys — guitar-friendly keys dominate'),
    height=350
)
chart

In [ ]:
# Duration trends over decades
duration_by_decade = df_tracks.groupby('decade')['duration_min'].agg(['mean', 'median']).reset_index()
duration_by_decade.columns = ['Decade', 'Mean Duration', 'Median Duration']
duration_by_decade = duration_by_decade[duration_by_decade['Decade'] >= 1920]

dur_melted = duration_by_decade.melt(id_vars='Decade', var_name='Metric', value_name='Duration (min)')

chart = alt.Chart(dur_melted).mark_line(strokeWidth=2.5, point=True).encode(
    x=alt.X('Decade:O', title='Decade'),
    y=alt.Y('Duration (min):Q', title='Duration (minutes)'),
    color=alt.Color('Metric:N', legend=alt.Legend(title='Metric')),
    tooltip=['Decade:O', 'Metric:N', alt.Tooltip('Duration (min):Q', format='.2f')]
).properties(
    title=alt.Title('Average Track Duration Over Decades',
                    subtitle='Songs got longer until the 1990s, then started shrinking in the streaming era'),
    height=350
)
chart

## 4.2 Advanced Visualizations


In [ ]:
# Pair plot using seaborn for multi-dimensional feature exploration
pair_features = ['danceability', 'energy', 'valence', 'acousticness', 'popularity']
pair_sample = df_tracks[pair_features + ['popularity_tier']].dropna().sample(3000, random_state=42)

fig = sns.pairplot(
    pair_sample, 
    hue='popularity_tier', 
    vars=pair_features[:4],
    palette='viridis',
    diag_kind='kde',
    plot_kws={'alpha': 0.4, 's': 15},
    height=2.5
)
fig.figure.suptitle('Pair Plot: Audio Features by Popularity Tier', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Genre landscape — 2D overview of genres by energy and danceability
genre_scatter_data = df_by_genres_clean.nlargest(50, 'popularity').copy()

chart = alt.Chart(genre_scatter_data).mark_circle().encode(
    x=alt.X('danceability:Q', title='Danceability'),
    y=alt.Y('energy:Q', title='Energy'),
    size=alt.Size('popularity:Q', legend=alt.Legend(title='Popularity'),
                  scale=alt.Scale(range=[50, 500])),
    color=alt.Color('valence:Q', scale=alt.Scale(scheme='redyellowgreen'),
                    legend=alt.Legend(title='Valence')),
    tooltip=['genres:N', 'danceability:Q', 'energy:Q', 'valence:Q', 'popularity:Q']
).properties(
    title=alt.Title('Genre Landscape: Energy, Danceability, Valence & Popularity',
                    subtitle='Size = popularity, Color = valence (green=happy, red=sad)'),
    height=450, width=550
)
chart

In [ ]:
# Heatmap: Average feature values across decades
decade_features = df_tracks.groupby('decade')[norm_features].mean()
decade_features = decade_features[decade_features.index >= 1920]
decade_melted = decade_features.reset_index().melt(
    id_vars='decade', var_name='Feature', value_name='Value'
)

chart = alt.Chart(decade_melted).mark_rect().encode(
    x=alt.X('decade:O', title='Decade'),
    y=alt.Y('Feature:N', title=None),
    color=alt.Color('Value:Q', scale=alt.Scale(scheme='viridis'),
                    legend=alt.Legend(title='Avg Value')),
    tooltip=['decade:O', 'Feature:N', alt.Tooltip('Value:Q', format='.3f')]
).properties(
    title=alt.Title('Audio Feature Heatmap Across Decades',
                    subtitle='Clear shift from acoustic/instrumental to electronic/danceable music'),
    height=300, width=500
)
chart

## 5. MODELING & PREDICTIONS

## 5.1 Build Predictive Models to Forecast Song Popularity

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
# Select features and target
feature_cols = ['danceability', 'energy', 'valence', 'acousticness', 
                'instrumentalness', 'liveness', 'speechiness', 'tempo', 
                'loudness', 'duration_min', 'key', 'mode']

model_df = df_tracks[feature_cols + ['popularity']].dropna()
X = model_df[feature_cols]
y = model_df['popularity']

# Split: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features for Linear Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set: {len(X_train):,} songs")
print(f"Testing set:  {len(X_test):,} songs")

In [ ]:
# Model 1: Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)
lr_pred = lr_model.predict(X_test_scaled)

# Model 2: Decision Tree
dt_model = DecisionTreeRegressor(max_depth=10, random_state=42)
dt_model.fit(X_train, y_train)
dt_pred = dt_model.predict(X_test)

print("Both models trained successfully!")

## 5.2 Evaluate the Performance of Different Models

In [ ]:
# Evaluate both models using predictive metrics
results = pd.DataFrame([
    {'Model': 'Linear Regression',
     'MAE': round(mean_absolute_error(y_test, lr_pred), 3),
     'RMSE': round(np.sqrt(mean_squared_error(y_test, lr_pred)), 3),
     'R² Score': round(r2_score(y_test, lr_pred), 4)},
    {'Model': 'Decision Tree',
     'MAE': round(mean_absolute_error(y_test, dt_pred), 3),
     'RMSE': round(np.sqrt(mean_squared_error(y_test, dt_pred)), 3),
     'R² Score': round(r2_score(y_test, dt_pred), 4)}
])

results

In [ ]:
# Model comparison chart
chart = alt.Chart(results).mark_bar(
    cornerRadiusTopLeft=4, cornerRadiusTopRight=4
).encode(
    x=alt.X('R² Score:Q', title='R² Score (Higher = Better)'),
    y=alt.Y('Model:N', title=None, sort='-x'),
    color=alt.Color('Model:N', scale=alt.Scale(range=['#1DB954', '#3498db']), legend=None),
    tooltip=['Model:N', 'MAE:Q', 'RMSE:Q', alt.Tooltip('R² Score:Q', format='.4f')]
).properties(
    title='Model Performance Comparison',
    height=150
)

chart

In [ ]:
# Actual vs Predicted plots
pred_df = pd.DataFrame({'Actual': y_test.values[:2000], 'LR': lr_pred[:2000], 'DT': dt_pred[:2000]})

line = alt.Chart(pd.DataFrame({'x': [0,100], 'y': [0,100]})).mark_line(
    strokeDash=[5,5], color='red', strokeWidth=2
).encode(x='x:Q', y='y:Q')

lr_chart = (alt.Chart(pred_df).mark_circle(size=10, opacity=0.3).encode(
    x=alt.X('Actual:Q', title='Actual Popularity'),
    y=alt.Y('LR:Q', title='Predicted Popularity')
) + line).properties(title='Linear Regression: Actual vs Predicted', height=350, width=380)

dt_chart = (alt.Chart(pred_df).mark_circle(size=10, opacity=0.3).encode(
    x=alt.X('Actual:Q', title='Actual Popularity'),
    y=alt.Y('DT:Q', title='Predicted Popularity')
) + line).properties(title='Decision Tree: Actual vs Predicted', height=350, width=380)

lr_chart | dt_chart

## 5.3 Fine-Tune Models to Improve Accuracy

In [ ]:
# Fine-tune Decision Tree — find the best depth
depth_results = []
for depth in [3, 5, 8, 10, 12, 15, 20]:
    dt = DecisionTreeRegressor(max_depth=depth, random_state=42)
    dt.fit(X_train, y_train)
    pred = dt.predict(X_test)
    depth_results.append({'Depth': depth, 
                          'R² Score': round(r2_score(y_test, pred), 4),
                          'MAE': round(mean_absolute_error(y_test, pred), 3)})

depth_df = pd.DataFrame(depth_results)
depth_df

In [ ]:
# Visualize depth tuning
chart = alt.Chart(depth_df).mark_line(
    strokeWidth=2.5, point=alt.OverlayMarkDef(size=80)
).encode(
    x=alt.X('Depth:Q', title='Tree Depth'),
    y=alt.Y('R² Score:Q', title='R² Score'),
    tooltip=['Depth:Q', alt.Tooltip('R² Score:Q', format='.4f'), 'MAE:Q']
).properties(
    title='Decision Tree: Finding Optimal Depth',
    height=300
)

chart

In [ ]:
# Train fine-tuned model with best depth
best_depth = int(depth_df.loc[depth_df['R² Score'].idxmax(), 'Depth'])
best_dt = DecisionTreeRegressor(max_depth=best_depth, random_state=42)
best_dt.fit(X_train, y_train)
best_pred = best_dt.predict(X_test)

# Final comparison
final_results = pd.DataFrame([
    {'Model': 'Linear Regression', 
     'MAE': round(mean_absolute_error(y_test, lr_pred), 3),
     'RMSE': round(np.sqrt(mean_squared_error(y_test, lr_pred)), 3),
     'R² Score': round(r2_score(y_test, lr_pred), 4)},
    {'Model': 'Decision Tree (Original)', 
     'MAE': round(mean_absolute_error(y_test, dt_pred), 3),
     'RMSE': round(np.sqrt(mean_squared_error(y_test, dt_pred)), 3),
     'R² Score': round(r2_score(y_test, dt_pred), 4)},
    {'Model': f'Decision Tree (Tuned, depth={best_depth})', 
     'MAE': round(mean_absolute_error(y_test, best_pred), 3),
     'RMSE': round(np.sqrt(mean_squared_error(y_test, best_pred)), 3),
     'R² Score': round(r2_score(y_test, best_pred), 4)}
])

print(f"Best depth: {best_depth}")
final_results

In [ ]:
# Final comparison chart
chart = alt.Chart(final_results).mark_bar(
    cornerRadiusTopLeft=4, cornerRadiusTopRight=4
).encode(
    x=alt.X('R² Score:Q', title='R² Score (Higher = Better)'),
    y=alt.Y('Model:N', title=None, sort='-x'),
    color=alt.Color('R² Score:Q', scale=alt.Scale(scheme='viridis'), legend=None),
    tooltip=['Model:N', 'MAE:Q', 'RMSE:Q', alt.Tooltip('R² Score:Q', format='.4f')]
).properties(
    title='Final Model Comparison After Fine-Tuning',
    height=200
)

chart

## 6. CONCLUSION

## 6.1 Key Findings

##### This analysis of 170,000+ Spotify tracks revealed several important patterns:

##### Music has evolved dramatically — Over the past century, music shifted from acoustic/instrumental to electronic, loud, and danceable. Acousticness dropped from ~0.9 to ~0.2.

##### The Loudness War is real — Average loudness increased from -25 dB to -7 dB, as producers competed for attention.

##### Happiness in music is declining — Valence peaked in the 1960s and has been steadily declining.

##### Popularity is driven by recency — Release year has the strongest association with popularity on Spotify.

##### Danceability drives engagement — T-test confirms popular tracks are significantly more danceable (p < 0.05).

##### Genre fingerprints are distinct — Audio features clearly differentiate genres.

##### Predictive Models — Linear Regression and Decision Tree models were built. Decision Tree outperformed Linear Regression. Predictive metrics (MAE, RMSE, R²) confirmed audio features can partially predict popularity.

##### The Hit Song Recipe — Popular songs have higher danceability (~0.67), energy (0.68), louder production (-6 dB), and lower acousticness (~0.15).

## 6.2 Implications

##### For Artists & Producers: Statistical analysis reveals that the "hit song recipe" includes high danceability, energy, and loud production with low acousticness. However, authenticity and genre-specific qualities remain important — niche genres have dedicated listener bases.

##### For Music Industry: The strong recency bias in Spotify's popularity metric suggests that catalog music is underrepresented. Strategies for re-promoting classic tracks could tap into underserved demand.

##### For Playlist Curators: Understanding the audio feature profiles of popular tracks can help in creating data-driven playlists that match listener preferences — high-energy, danceable, and vocally-driven tracks perform best.

##### For Data Analysts: Audio features show weak-to-moderate individual correlations with popularity, confirming that popularity depends on many external factors (artist fame, marketing, playlist placement, viral moments) beyond just the sound itself.

## 6.3 Future Research Directions

##### Sentiment Analysis on Lyrics — Incorporate NLP analysis of song lyrics to understand the relationship between lyrical content and popularity

##### Time-Series Forecasting — Analyze how audio feature trends will continue evolving over the next decade

##### Playlist Analysis — Study how playlist placement affects track popularity and whether certain audio features lead to higher playlist inclusion

##### Regional Preferences — Compare listening habits across different countries and cultures to identify regional music preferences

##### Cross-Platform Comparison — Compare Spotify data with Apple Music, YouTube Music, and other platforms to identify platform-specific listener preferences

##### Artist Growth Analysis — Track how artists' audio feature profiles change over their careers and whether this correlates with growing or declining popularity